In [91]:
pip install mlflow dagshub

In [92]:
!pip install nltk

### Set Mlflow

In [93]:
import dagshub
dagshub.init(repo_owner='iamdebasishdas123', repo_name='YouTube-Mood-Tracker', mlflow=True)
import mlflow

Initialized MLflow to track repo "iamdebasishdas123/YouTube-Mood-Tracker"

Repository iamdebasishdas123/YouTube-Mood-Tracker initialized!

## Load Dataset

In [94]:
import numpy as np
import pandas as pd

In [95]:
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


## 1. Preprocessing Data

In [96]:
df.sample(20).to_dict(orient='records')

[{'clean_comment': 'jaan maar denge ', 'category': 0},
 {'clean_comment': 'fuck this shit ragaism coming soon', 'category': -1},
 {'clean_comment': ' mean has number indianfood using cbd ', 'category': -1},
 {'clean_comment': ' ', 'category': 0},
 {'clean_comment': ' all wish for this someday ', 'category': 0},
 {'clean_comment': nan, 'category': 0},
 {'clean_comment': 'mitron aur landoor chutyaaa bannaya badda mazza ayya chutya bannayegaaaa aur mazza ayeenge ',
  'category': 0},
 {'clean_comment': ' team ppr bonuses pick one bmarsh mcrabtree chyde mthomas ',
  'category': 0},
 {'clean_comment': 'banda bundaas hain his speech was virat kohli hitting triple',
  'category': 0},
 {'clean_comment': 'shazia ilmi loses 340 votes ', 'category': -1},
 {'clean_comment': 'nepanašu kad daug pastangų įdėjai post pavadinimą galvodamas',
  'category': 0},
 {'clean_comment': 'and when the free encyclopedia the team sports and other news items credits sr33 new delhi the team sports and other news item

In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37249 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   clean_comment  37149 non-null  object
 1   category       37249 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 582.1+ KB


In [98]:
df['category'].value_counts()

,count
category,
1,15830
0,13142
-1,8277


In [99]:
df.duplicated().sum()

np.int64(449)

- Problem of Database
1. backslash in the data
2. Null data arund 100 rows
3. 449 duplicated data
3. Imbalanced data 

##### 1.1 Remove NA data

In [100]:
df.dropna(inplace=True)

##### 1.2 Remove Duplicate

In [101]:
df.drop_duplicates(inplace=True)

In [102]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36799 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   clean_comment  36799 non-null  object
 1   category       36799 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 862.5+ KB


In [103]:
df = df[~(df['clean_comment'].str.strip() == '')]

In [104]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36793 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   clean_comment  36793 non-null  object
 1   category       36793 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 862.3+ KB


In [105]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [106]:
# Ensure necessary NLTK data is downloaded
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [107]:
# Define the preprocessing function
def preprocess_comment(comment):
    # Convert to lowercase
    comment = comment.lower()

    # Remove trailing and leading whitespaces
    comment = comment.strip()

    # Remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # Remove non-alphanumeric characters, except punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # Remove stopwords but retain important ones for sentiment analysis
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

    return comment

In [108]:
# Apply the preprocessing function to the 'clean_comment' column
df['clean_comment'] = df['clean_comment'].apply(preprocess_comment)

In [109]:
df.sample(20).to_dict(orient='records')

[{'clean_comment': 'rula diya come back 2019 maybe asshats deserve hero need',
  'category': 0},
 {'clean_comment': 'pretty sure direct violation license section iii distribute publicly perform work adaptation collection must provide reasonable medium mean utilizing iii extent reasonably practicable uri licensor specifies associated work',
  'category': 1},
 {'clean_comment': 'cant watch video office hope someone start livebloging',
  'category': 0},
 {'clean_comment': 'fucking manmohan bow front mother sonia namo front mother india spot difference sorry draw parallel dolt resist',
  'category': -1},
 {'clean_comment': 'faasos blaming release piece foreign location based intern already fired',
  'category': -1},
 {'clean_comment': 'muslim used rule entire subcontinent british hmm cogent people certainly',
  'category': 1},
 {'clean_comment': 'hello american came across headline html headline indian state election matter whole world but article nothing describe part anyone maybe give in

## 2. Model Building

In [110]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

In [111]:
# Step 1: Vectorize the comments using Bag of Words (CountVectorizer)
vectorizer = CountVectorizer(max_features=5000)  # Bag of Words model with a limit of 5000 features

In [112]:
X = vectorizer.fit_transform(df['clean_comment']).toarray()
y = df['category']  # Assuming 'sentiment' is the target variable (0 or 1 for binary classification)

In [113]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [114]:
X.shape

(36793, 5000)

In [115]:

y = y.map({-1: 0, 0: 1, 1: 2})


In [116]:
y.shape

(36793,)

In [117]:
# Set or create an experiment
mlflow.set_experiment("Test Experiment Youtube Mood Tracker")

<Experiment: artifact_location='mlflow-artifacts:/a919a8ba2f0a451c8fdec499a9ec20b4', creation_time=1778520588352, experiment_id='0', last_update_time=1778520588352, lifecycle_stage='active', name='Test Experiment Youtube Mood Tracker', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [118]:
import seaborn as sns

In [119]:
# Step 1: Split the data into training and testing sets (80% train, 20% test)
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost  # Crucial for logging XGBoost correctly
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Step 1: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: Define and train an XGBoost baseline model
with mlflow.start_run() as run:
    # Metadata tags
    mlflow.set_tag("mlflow.runName", "XGBoost_Baseline_TrainTestSplit")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("model_type", "XGBClassifier")
    mlflow.set_tag(
        "description",
        "Baseline XGBoost sentiment model using Bag of Words (BoW)",
    )

    # Log vectorizer params
    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("vectorizer_max_features", vectorizer.max_features)

    # Model Hyperparameters
    params = {
        "n_estimators": 350,
        "max_depth": 12,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "eval_metric": "logloss",
    }
    mlflow.log_params(params)  # Dictionary logging keeps code clean

    # Train model
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Classification report logic
    classification_rep = classification_report(y_test, y_pred, output_dict=True)
    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            # Formats keys cleanly as "class_0_precision", "macro_avg_f1-score"
            clean_label = label.replace(" ", "_")
            for metric, value in metrics.items():
                mlflow.log_metric(f"{clean_label}_{metric}", value)

    # Confusion Matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")

    # Consistent local paths to prevent file-not-found errors
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.close()  # Close the plot to free memory

    # Native XGBoost logging
    mlflow.xgboost.log_model(model, "xgboost_model")

    # Save data snippet
    df.to_csv("dataset.csv", index=False)
    mlflow.log_artifact("dataset.csv")

print(f"Accuracy: {accuracy}")


2026/05/18 17:22:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost_Baseline_TrainTestSplit at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0/runs/d41936a8fcde4c67b18cfb94565807ff
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0
Accuracy: 0.8498437287674956


In [120]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.69      0.76      1650
           1       0.82      0.97      0.89      2555
           2       0.89      0.84      0.86      3154

    accuracy                           0.85      7359
   macro avg       0.85      0.83      0.83      7359
weighted avg       0.85      0.85      0.85      7359



### Run Experiments


##### 1.1 BOW or TFIDF

- Bag of Word(BOW)
1. It is a NLP technique to create embedding
2. Create unique collection of words from the all documents.
3. Count how many times each word from the master vocabulary appears in a specific document.
4. Represent each document as an array (vector) of these word counts.

- Example
Document 1: "The dog barked at the cat.

Document 2: "The cat sat on the mat."First

compile the unique vocabulary: ['the', 'dog', 'barked', 'at', 'cat', 'sat', 'on', 'mat'] (Total length: 8).

Using BoW, we represent the documents as numerical frequency vectors:

Document 1: \([2, 1, 1, 1, 1, 0, 0, 0]\)

Document 2: \([2, 0, 0, 0, 1, 1, 1, 1]\)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

def compute_bow(corpus, max_features=None):
    """
    Converts a text corpus into a Bag of Words matrix.
    """
    # Initialize the vectorizer
    vectorizer = CountVectorizer(max_features=max_features)
    
    # Learn vocabulary and transform text to sparse matrix
    bow_sparse = vectorizer.fit_transform(corpus)
    
    # Convert to a readable DataFrame with word labels
    feature_names = vectorizer.get_feature_names_out()
    bow_df = pd.DataFrame(bow_sparse.toarray(), columns=feature_names)
    
    return bow_df, vectorizer


- TF-IDF (Term Frequency-Inverse Document Frequency) : TF-IDF (Term Frequency-Inverse Document Frequency) is a foundational natural language processing algorithm that measures how important a word is to a specific document within a collection of text [1, 2]. It significantly improves upon the classic Bag of Words model because it does not just count how often words appear; it penalises words that show up everywhere (like "the", "is", or "and") and rewards unique words that convey actual meaning [1, 2].

1. Term Frequency (TF): Measures how frequently a word appears inside a single document.
2. Inverse Document Frequency (IDF): Measures how unique or rare a word is across your entire collection of documents (corpus).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

def compute_tfidf(corpus, max_features=None):
    """
    Converts a text corpus into a TF-IDF matrix.
    """
    # Initialize the vectorizer
    vectorizer = TfidfVectorizer(max_features=max_features)
    
    # Learn vocabulary and weights, transform text to sparse matrix
    tfidf_sparse = vectorizer.fit_transform(corpus)
    
    # Convert to a readable DataFrame with word labels
    feature_names = vectorizer.get_feature_names_out()
    tfidf_df = pd.DataFrame(tfidf_sparse.toarray(), columns=feature_names)
    
    return tfidf_df, vectorizer


##### 1.2 Model Selection